In [ ]:
# Importing all of the things
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.dummy import DummyRegressor

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Read in the data
file_path = "data/RRCA_baseflow.csv"

df = pd.read_csv(file_path)

# Clean whitespace from column names
df.columns = df.columns.str.strip()

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())
display(df.info())

# Check for missing columns
df = df.rename(columns={
    "Date": "date_days",
    "Segment_id": "segment_id",
    "x": "x",
    "y": "y",
    "Evapotranspiration": "evapotranspiration",
    "Precipitation": "precipitation",
    "Irrigation_pumping": "irrigation_pumping",
    "Observed": "observed"
})

expected_columns = [
    "date_days",
    "segment_id",
    "x",
    "y",
    "evapotranspiration",
    "precipitation",
    "irrigation_pumping",
    "observed"
]

missing_columns = set(expected_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

display(df.head())

In [ ]:
# Fix the dates
df["date_original"] = pd.to_numeric(df["date_days"], errors="coerce")

df["days_since_1900"] = df["date_original"] - 693963

df["date"] = (
    pd.Timestamp("1900-01-01")
    + pd.to_timedelta(df["days_since_1900"], unit="D")
)

display(
    df[["date_original", "days_since_1900", "date"]].head(10)
)

print("Date range:", df["date"].min(), "to", df["date"].max())
print("Unique dates:", df["date"].nunique())

In [ ]:
# Verify data types
print("Data types:")
print(df.dtypes)

# Check for missing and duplicate values
print("\nMissing values:")
display(
    df[expected_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nDuplicate complete rows:", df.duplicated().sum())
print("\nUnique river segments:", df["segment_id"].nunique())
print("\nObservations per segment:")
display(
    df["segment_id"]
    .value_counts()
    .describe()
    .to_frame("observations_per_segment")
)

# Describe the data
print("\nNumeric summary:")
display(df[[
    "days_since_1900",
    "x", "y",
    "evapotranspiration",
    "precipitation",
    "irrigation_pumping",
    "observed"
]].describe().T)

In [ ]:
# Examine the distribution and potential outliers of observed baseflow.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(
    data=df, x="observed", kde=True, ax=axes[0]
)
axes[0].set_title("Distribution of observed baseflow")

sns.boxplot(
    data=df, x="observed", ax=axes[1]
)
axes[1].set_title("Observed baseflow: potential outliers")

plt.tight_layout()
plt.show()

print(df["observed"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

In [ ]:
# Examine the relationships between predictors and baseflow
predictors = [
    "days_since_1900",
    "x",
    "y",
    "evapotranspiration",
    "precipitation",
    "irrigation_pumping"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, predictors):
    sns.scatterplot(
        data=df,
        x=col,
        y="observed",
        alpha=0.35,
        s=18,
        ax=ax
    )
    ax.set_title(f"Observed baseflow vs. {col}")

plt.tight_layout()
plt.show()

In [ ]:
# Check correlations
numeric_cols = [
    "days_since_1900",
    "x", "y",
    "evapotranspiration",
    "precipitation",
    "irrigation_pumping",
    "observed"
]

corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1
)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Patterns over time
monthly = (
    df.dropna(subset=["date", "observed"])
      .groupby("date", as_index=False)
      .agg(
          mean_observed=("observed", "mean"),
          median_observed=("observed", "median"),
          std_observed=("observed", "std"),
          n_segments=("observed", "size")
      )
      .sort_values("date")
)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    monthly["date"], monthly["mean_observed"],
    label="Monthly mean", alpha=0.8
)
ax.plot(
    monthly["date"], monthly["median_observed"],
    label="Monthly median", alpha=0.8
)

ax.set_title("Baseflow over time across all observations")
ax.set_xlabel("Date")
ax.set_ylabel("Observed baseflow")
ax.legend()

plt.tight_layout()
plt.show()

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

# Check seasonality
plt.figure(figsize=(11, 5))
sns.boxplot(data=df, x="month", y="observed")
plt.title("Baseflow distribution by calendar month")
plt.xlabel("Month")
plt.ylabel("Observed baseflow")
plt.tight_layout()
plt.show()

# Year-to-year changes
plt.figure(figsize=(14, 5))
sns.lineplot(
    data=df,
    x="year",
    y="observed",
    estimator="mean",
    errorbar=None
)
plt.title("Mean observed baseflow by year")
plt.xlabel("Year")
plt.ylabel("Mean baseflow")
plt.tight_layout()
plt.show()

In [ ]:
# Comparing individual river segments
segment_summary = (
    df.groupby("segment_id")
      .agg(
          n=("observed", "count"),
          mean_baseflow=("observed", "mean"),
          median_baseflow=("observed", "median"),
          std_baseflow=("observed", "std"),
          mean_precipitation=("precipitation", "mean"),
          mean_evapotranspiration=("evapotranspiration", "mean"),
          mean_pumping=("irrigation_pumping", "mean"),
          x=("x", "median"),
          y=("y", "median")
      )
      .sort_values("mean_baseflow")
)

display(segment_summary)

segment_order = (
    df.groupby("segment_id")["observed"]
      .median()
      .sort_values()
      .index
)

plt.figure(figsize=(max(12, len(segment_order) * 0.45), 6))

sns.boxplot(
    data=df,
    x="segment_id",
    y="observed",
    order=segment_order
)

plt.title("Baseflow distribution by river segment")
plt.xlabel("Segment ID (ordered by median baseflow)")
plt.ylabel("Observed baseflow")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Time trends for individual segments
top_segments = (
    df["segment_id"]
    .value_counts()
    .head(8)
    .index
)

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=df[df["segment_id"].isin(top_segments)],
    x="date",
    y="observed",
    hue="segment_id",
    estimator="mean",
    errorbar=None
)

plt.title("Baseflow over time for eight frequently observed segments")
plt.xlabel("Date")
plt.ylabel("Observed baseflow")
plt.legend(title="Segment ID", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Patterns in spatial locations
plt.figure(figsize=(9, 7))

sc = plt.scatter(
    df["x"],
    df["y"],
    c=df["observed"],
    cmap="viridis",
    alpha=0.7,
    s=25
)

plt.colorbar(sc, label="Observed baseflow")
plt.xlabel("x coordinate")
plt.ylabel("y coordinate")
plt.title("Spatial distribution of observed baseflow")
plt.axis("equal")
plt.tight_layout()
plt.show()

segment_locations = (
    df.groupby("segment_id")
      .agg(
          x=("x", "median"),
          y=("y", "median"),
          mean_baseflow=("observed", "mean")
      )
      .dropna()
)

plt.figure(figsize=(9, 7))

sc = plt.scatter(
    segment_locations["x"],
    segment_locations["y"],
    c=segment_locations["mean_baseflow"],
    cmap="viridis",
    s=70,
    edgecolor="black",
    linewidth=0.3
)

plt.colorbar(sc, label="Mean baseflow per segment")
plt.xlabel("x coordinate")
plt.ylabel("y coordinate")
plt.title("Spatial pattern of mean baseflow by segment")
plt.axis("equal")
plt.tight_layout()
plt.show()